In [ ]:
# Ô 1 — Cài thư viện + nối Google Drive (chạy lại mỗi lần mở Colab)
import os, sys, importlib
try:
    import google.colab  # chỉ có khi chạy trên Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install tldextract rapidfuzz shap xgboost
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/AIC211_URL'   # thư mục chung của nhóm
else:
    ROOT = os.path.abspath('AIC211_URL')          # chạy thử trên máy (không phải Colab)

for d in ['data/raw', 'data/processed', 'src', 'models', 'results', 'figures', 'notebooks']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
sys.path.insert(0, f'{ROOT}/src')
print('ROOT =', ROOT)
print(sorted(os.listdir(ROOT)))

In [ ]:
# Ô 1b — Ghi lại src/e3.py (bản nhanh): tấn công ngẫu nhiên, tấn công tham lam hộp đen, score_fn, ASR
E3_CODE = r"""
import random
import numpy as np, pandas as pd
from common import normalize_url, split_parts, get_scores, make_F1
from attacks import apply_op, is_valid

FEATURIZE = {'F1': make_F1, 'F2': lambda d: d[['host', 'rest']]}

def make_score_fn(model, featurize, preprocess=None):
    # Hộp đen: nhận list URL -> mảng điểm độc. preprocess = phòng thủ D1 (None = không phòng thủ)
    def score_fn(urls):
        u = [preprocess(x) if preprocess else x for x in urls]
        d = pd.DataFrame({'url_norm': [normalize_url(x) for x in u]})
        parts = d['url_norm'].map(split_parts)
        d['host'] = parts.str[0]; d['rest'] = parts.str[1]
        return get_scores(model, featurize(d))
    return score_fn

def random_attack(url, k, ops, rng, benign_domains):
    # Kẻ tấn công không biết gì: chọn bừa k biến đổi hợp lệ
    cur, applied, tries = url, [], 0
    while len(applied) < k and tries < 20 * max(k, 1):
        tries += 1
        op = rng.choice(ops)
        new = apply_op(op, cur, rng)
        if is_valid(cur, new, op, benign_domains):
            cur, applied = new, applied + [op]
    return cur, applied

def greedy_attack(url, score_fn, thr, k, ops, rng, benign_domains, n_cand=3):
    # Hộp đen có truy vấn: mỗi bước thử mọi loại biến đổi (n_cand cách mỗi loại), giữ cách làm điểm giảm nhiều nhất
    cur = url; s = score_fn([cur])[0]; queries = 1; applied = []
    for _ in range(k):
        if s <= thr: break                       # đã lọt -> dừng
        cands = []
        for op in ops:
            for _ in range(n_cand):
                new = apply_op(op, cur, rng)
                if is_valid(cur, new, op, benign_domains):
                    cands.append((op, new))
        if not cands: break
        sc = score_fn([c[1] for c in cands]); queries += len(cands)
        j = int(np.argmin(sc))
        if sc[j] >= s: break                     # không giảm được nữa
        cur, s = cands[j][1], sc[j]; applied.append(cands[j][0])
    return {'adv_url': cur, 'ops': applied, 'queries': queries}

def greedy_attack_batch(urls, score_fn, thr, k, ops, rng, benign_domains, n_cand=3):
    # Giống greedy_attack nhưng chấm điểm CẢ LÔ URL trong 1 lần gọi (nhanh hơn rất nhiều với Random Forest)
    n = len(urls); cur = list(urls); s = np.asarray(score_fn(cur), dtype=float)
    queries = [1] * n; applied = [[] for _ in range(n)]
    active = s > thr
    for _ in range(k):
        cands = []                                   # (chỉ số URL, op, URL mới)
        for i in np.where(active)[0]:
            for op in ops:
                for _ in range(n_cand):
                    new = apply_op(op, cur[i], rng)
                    if is_valid(cur[i], new, op, benign_domains):
                        cands.append((i, op, new))
        if not cands: break
        sc = np.asarray(score_fn([c[2] for c in cands]), dtype=float)
        best = {}
        for (i, op, new), v in zip(cands, sc):
            queries[i] += 1
            if i not in best or v < best[i][0]:
                best[i] = (v, op, new)
        for i in np.where(active)[0]:
            if i in best and best[i][0] < s[i]:
                s[i] = best[i][0]; cur[i] = best[i][2]; applied[i].append(best[i][1])
            else:
                active[i] = False                    # không giảm được nữa
        active &= s > thr                            # đã lọt -> dừng
    return [{'adv_url': cur[i], 'ops': applied[i], 'queries': queries[i]} for i in range(n)]

def attack_success_rate(mal_urls, score_fn, thr, k, ops, mode, benign_domains, seed=0):
    # ASR = trong số URL độc BAN ĐẦU bị bắt, bao nhiêu % lọt sau khi biến đổi
    rng = random.Random(seed)
    base = score_fn(mal_urls)
    detected = [u for u, s in zip(mal_urls, base) if s > thr]
    if mode == 'random':
        rows = []
        for u in detected:
            adv, applied = random_attack(u, k, ops, rng, benign_domains)
            rows.append({'adv_url': adv, 'ops': applied, 'queries': 0})
    else:
        rows = greedy_attack_batch(detected, score_fn, thr, k, ops, rng, benign_domains) if detected else []
    res = pd.DataFrame(rows)
    if len(res):
        res['evaded'] = score_fn(list(res['adv_url'])) <= thr
    asr = float(res['evaded'].mean()) if len(res) else 0.0
    n_evaded = int(res['evaded'].sum()) if len(res) else 0
    return {'asr': asr,
            'recall_clean': len(detected) / max(len(mal_urls), 1),
            'recall_attack': (len(detected) - n_evaded) / max(len(mal_urls), 1),
            'n_detected': len(detected),
            'mean_queries': float(res['queries'].mean()) if len(res) else 0.0}, res
"""
with open(f'{ROOT}/src/e3.py', 'w', encoding='utf-8') as f:
    f.write(E3_CODE)
import e3; importlib.reload(e3)
from e3 import *
import common; importlib.reload(common)
print('Đã ghi', f'{ROOT}/src/e3.py')

In [ ]:
# Ô 2 — Nạp code (common, attacks, e3, defenses), dữ liệu, model và ngưỡng đã đóng băng
for f in ['models/thresholds.json', 'models/RF_F1.joblib', 'models/LR_F2.joblib', 'src/e3.py', 'src/defenses.py']:
    assert os.path.exists(f'{ROOT}/{f}'), f'Thiếu {f}: chạy xong tuần 5 (Trí: TV4 tuần 5, Nhựt: TV3 tuần 5) trước.'
import common; importlib.reload(common)
from common import *
import json, joblib, random, time
import matplotlib.pyplot as plt
A = pd.read_parquet(f'{ROOT}/data/processed/A.parquet')
B = pd.read_parquet(f'{ROOT}/data/processed/B.parquet')
THR = json.load(open(f'{ROOT}/models/thresholds.json'))
INFO = json.load(open(f'{ROOT}/models/model_info.json'))
tr, va, te = (A.split == 'train').values, (A.split == 'val').values, (A.split == 'test').values
import attacks, e3, defenses
for m in (attacks, e3, defenses): importlib.reload(m)
from attacks import *
from e3 import *
from defenses import d1_normalize

MODELS = {name: joblib.load(f'{ROOT}/models/{name}.joblib') for name in ['RF_F1', 'LR_F2']}
SCORE = {name: make_score_fn(m, FEATURIZE[INFO[name]['features']]) for name, m in MODELS.items()}
benign_domains = set(A[A.label == 0]['reg_domain'])
print('Ngưỡng (đóng băng):', {k: round(THR[k], 4) for k in MODELS})

In [ ]:
# Ô 3 — Chọn URL độc để tấn công: N URL từ A-test và N URL từ B (cố định seed để lần sau giống nhau)
N = 300          # Colab chậm thì giảm xuống 200; còn dư thời gian thì tăng 500
rng0 = random.Random(0)
MAL = {
    'A-test': rng0.sample([to_attack_form(u) for u in A[te & (A.label == 1).values]['url_norm']], N),
    'B':      rng0.sample([to_attack_form(u) for u in B[B.label == 1]['url_norm']], N),
}
OPS = {'free': FREE_OPS, 'all': FREE_OPS + PAID_OPS}
print({k: len(v) for k, v in MAL.items()}, '| ops:', OPS)

In [ ]:
# Ô 4 — E3 ĐẦY ĐỦ (seed 0): 2 model × k {0,1,2,3,5} × ngẫu nhiên/tham lam × miễn phí/tất cả × A-test/B
# Mất khoảng 15–40 phút trên Colab. Chạy 1 lần, kết quả tự ghi vào results.csv.
rows, t0 = [], time.time()
for model in MODELS:
    for src, urls in MAL.items():
        for mode in ['random', 'greedy']:
            for ops_name, ops in OPS.items():
                for k in [0, 1, 2, 3, 5]:
                    m, _ = attack_success_rate(urls, SCORE[model], THR[model], k, ops, mode, benign_domains, seed=0)
                    log_result(ROOT, 'E3', model, src, m, seed=0, k=k, ops=ops_name, mode=mode)
                    rows.append({'model': model, 'nguồn': src, 'kiểu': mode, 'biến đổi': ops_name, 'k': k, **m})
        print(f'{model} xong ({(time.time() - t0) / 60:.1f} phút)')
E3 = pd.DataFrame(rows)
E3.to_csv(f'{ROOT}/results/E3_seed0.csv', index=False)
print(E3.pivot_table(index=['model', 'nguồn', 'kiểu', 'biến đổi'], columns='k', values='recall_attack').round(3).to_string())

In [ ]:
# Ô 5 — ASR theo TỪNG biến đổi (k = 1, tấn công tham lam, A-test): biến đổi nào mạnh nhất? (trả lời H2)
rows = []
for model in MODELS:
    for op in FREE_OPS + PAID_OPS:
        m, _ = attack_success_rate(MAL['A-test'], SCORE[model], THR[model], 1, [op], 'greedy', benign_domains, seed=0)
        log_result(ROOT, 'E3_perT', model, 'A-test', m, seed=0, k=1, ops=op, mode='greedy')
        rows.append({'model': model, 'T': op, 'ASR': m['asr']})
perT = pd.DataFrame(rows).pivot(index='T', columns='model', values='ASR')
perT.to_csv(f'{ROOT}/results/E3_asr_per_T.csv')
print(perT.round(3).to_string())

In [ ]:
# Ô 6 — HÌNH CHÍNH của bài: recall dưới tấn công theo k (F1 vs F2, miễn phí vs tất cả)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, model in zip(axes, MODELS):
    for (mode, ops_name), style in {('random', 'free'): ':', ('random', 'all'): '--',
                                     ('greedy', 'free'): '-', ('greedy', 'all'): '-'}.items():
        for src, color in [('A-test', '#3b6ea5'), ('B', '#c2552d')]:
            d = E3[(E3.model == model) & (E3['nguồn'] == src) & (E3['kiểu'] == mode) & (E3['biến đổi'] == ops_name)]
            ax.plot(d.k, d.recall_attack, style, color=color, marker='o' if ops_name == 'all' else '.',
                    lw=2 if mode == 'greedy' else 1.2, label=f'{src} · {mode} · {ops_name}')
    ax.set_title(model); ax.set_xlabel('số biến đổi tối đa k'); ax.set_xticks([0, 1, 2, 3, 5]); ax.grid(alpha=.3)
axes[0].set_ylabel('recall ở FPR 1% (dưới tấn công)')
axes[1].legend(fontsize=7, loc='upper right')
plt.tight_layout(); plt.savefig(f'{ROOT}/figures/E3_recall_vs_k.png', dpi=200); plt.show()

In [ ]:
# Ô 6b — Sửa D1: URL hỏng (vd. có dấu [ ] lạ) thì giữ nguyên, không báo lỗi
DEF_CODE = r"""
import re, unicodedata
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode
from attacks import HOMOGLYPH

CONFUSABLE = {v: k for k, v in HOMOGLYPH.items()}   # bản rút gọn; báo cáo ghi rõ là bảng nhỏ
TRACKING = re.compile(r'^(utm_\w+|ref|fbclid|gclid|mc_cid|mc_eid)$')

def d1_normalize(url):
    # Giải punycode -> Unicode -> đổi ký tự giả (Cyrillic) về Latin; bỏ tham số tracking
    try:
        p = urlsplit(url if '://' in url else 'http://' + url)
        host = p.hostname or ''
    except ValueError:
        return url                     # URL hỏng: giữ nguyên
    labels = []
    for lab in host.split('.'):
        if lab.startswith('xn--'):
            try: lab = lab.encode('ascii').decode('idna')
            except UnicodeError: pass
        lab = unicodedata.normalize('NFKC', lab)
        labels.append(''.join(CONFUSABLE.get(ch, ch) for ch in lab))
    q = [(k, v) for k, v in parse_qsl(p.query, keep_blank_values=True) if not TRACKING.match(k.lower())]
    return urlunsplit((p.scheme, '.'.join(labels), p.path, urlencode(q), ''))
"""
with open(f'{ROOT}/src/defenses.py', 'w', encoding='utf-8') as f:
    f.write(DEF_CODE)
import defenses; importlib.reload(defenses)
from defenses import d1_normalize
print(d1_normalize('http://abc[.com/x'), '|', d1_normalize('http://xn--acount-blog16-p8k.top/a?utm_source=x&id=1'))

In [ ]:
# Ô 7 — D1 CHÍNH THỨC: train lại RF-F1 và LR-F2 trên URL đã qua d1_normalize (cả lúc train lẫn lúc chấm)
from sklearn.base import clone

def d1_frame(df):
    # Áp D1 lên URL rồi chuẩn hóa như thường -> DataFrame có url_norm, host, rest, label
    u = [normalize_url(d1_normalize(to_attack_form(x))) for x in df['url_norm']]
    out = pd.DataFrame({'url_norm': u, 'label': df['label'].values})
    parts = out['url_norm'].map(split_parts)
    out['host'] = parts.str[0]; out['rest'] = parts.str[1]
    return out

A1, B1 = d1_frame(A), d1_frame(B)
THR_D = {}
for name in MODELS:
    fs = INFO[name]['features']
    m = clone(MODELS[name]).fit(FEATURIZE[fs](A1[tr]), A1.label[tr])
    THR_D[f'{name}_D1'] = threshold_at_fpr(A1.label[va], get_scores(m, FEATURIZE[fs](A1[va])))
    joblib.dump(m, f'{ROOT}/models/{name}_D1.joblib')
    for src, d in [('A-test', A1[te]), ('B', B1)]:
        met = evaluate(d.label, get_scores(m, FEATURIZE[fs](d)), THR_D[f'{name}_D1'])
        log_result(ROOT, 'E4_clean', name, src, met, defense='D1')
        print(f'{name}+D1 sạch · {src:6s}: F1 = {met["f1"]:.3f} | recall = {met["recall"]:.3f} | FPR = {met["fpr"]:.4f}')
json.dump(THR_D, open(f'{ROOT}/models/thresholds_D1.json', 'w'), indent=2)   # file riêng, KHÔNG đụng thresholds.json

In [ ]:
# Ô 8 — D1 dưới tấn công: so ASR không phòng thủ vs có D1 (tham lam, k = 3, tất cả biến đổi, A-test)
rows = []
for name in MODELS:
    fs = INFO[name]['features']
    m_d1 = joblib.load(f'{ROOT}/models/{name}_D1.joblib')
    score_d1 = make_score_fn(m_d1, FEATURIZE[fs], preprocess=d1_normalize)
    for label, sf, thr in [('không phòng thủ', SCORE[name], THR[name]), ('D1', score_d1, THR_D[f'{name}_D1'])]:
        m, _ = attack_success_rate(MAL['A-test'], sf, thr, 3, FREE_OPS + PAID_OPS, 'greedy', benign_domains, seed=0)
        log_result(ROOT, 'E4_attack', name, 'A-test', m, defense='none' if label != 'D1' else 'D1',
                   seed=0, k=3, ops='all', mode='greedy')
        rows.append({'model': name, 'phòng thủ': label, **m})
print(pd.DataFrame(rows)[['model', 'phòng thủ', 'recall_clean', 'recall_attack', 'asr', 'mean_queries']].round(3).to_string())